# Ablation: MLM pretraining × pooling — joint Sinhala sentiment + intent

| Variant | MLM stage | Pooling |
|---|---|---|
| `MLM+attn` | yes | task-specific multi-head attention pooling |
| `noMLM+attn` | no | task-specific multi-head attention pooling |
| `MLM+mean` | yes | masked mean pooling (shared vector) |
| `noMLM+mean` | no | masked mean pooling (shared vector) |

**Controlled:** every hyperparameter in the config cell is shared by all variants; same split, same class weights, same seeds, same batch order per seed.
**Splits:** test ≥ 300 rows (manually annotated, blind to silver labels) · val (silver) for early stopping · train (silver).
**MLM corpus:** train + val text only (+ optional unlabeled extra). Test text is never seen by MLM.
**Workflow:** run §1–3 → annotate `annotation_sheet.csv` → run §4 (training; does not need gold) → run §5 after gold file exists.

## 1. Config

In [1]:
# !pip install -q transformers accelerate
import os, re, gc, math, json, random, time, itertools, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModel, AutoModelForMaskedLM,
                          DataCollatorForLanguageModeling, get_linear_schedule_with_warmup)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, cohen_kappa_score
warnings.filterwarnings('ignore')

# ── Hyperparameters: IDENTICAL for all variants ───────────────────────────────
MODEL_NAME     = 'xlm-roberta-base'
MAX_LEN        = 128
MLM_PROB       = 0.15
MLM_EPOCHS     = 3
MLM_BATCH      = 8
MLM_GRAD_ACC   = 2
MLM_LR         = 2e-5
FT_EPOCHS      = 10
FT_BATCH       = 8
FT_GRAD_ACC    = 2
FT_LR          = 2e-5
ENC_LR_MULT    = 0.1          # encoder LR = FT_LR * ENC_LR_MULT
WEIGHT_DECAY   = 0.01
DROPOUT        = 0.3
NUM_ATTN_HEADS = 4
PATIENCE       = 3            # early stopping on val mean macro-F1 (sent, intent)
SEEDS          = [42, 43, 44]
POOLINGS       = ['attn', 'mean']      # add 'cls' to also run CLS-pooling variants

# ── Data / split ──────────────────────────────────────────────────────────────
CSV_PATH     = '/content/sinhala_silver_labels.csv'
TEST_N       = 300            # must be >= 300
VAL_FRAC     = 0.15           # fraction of the non-test pool
SPLIT_SEED   = 42

# Optional: extra UNLABELED text for MLM (e.g. rows from the 641K corpus). None = train+val only.
MLM_EXTRA_CSV = None          # e.g. '/kaggle/input/.../full_corpus.csv'
MLM_EXTRA_COL = 'Sentence'
MLM_EXTRA_N   = 20000

FORCE_CPU = False

# ── Paths ─────────────────────────────────────────────────────────────────────
WORK      = '/content/' if os.path.exists('/content/') else '.'
SPLIT_DIR = f'{WORK}/splits'
PRED_DIR  = f'{WORK}/preds'
ENC_DIR   = f'{WORK}/mlm_encoders'
ANNOT_CSV = f'{WORK}/annotation_sheet.csv'        # exported for you to annotate
GOLD_CSV  = f'{WORK}/annotation_sheet_done.csv'   # upload your completed sheet here
for d in (SPLIT_DIR, PRED_DIR, ENC_DIR): os.makedirs(d, exist_ok=True)

assert TEST_N >= 300, 'test set must have at least 300 rows'

VARIANTS = {f"{'MLM' if m else 'noMLM'}+{p}": (m, p) for p in POOLINGS for m in (True, False)}

def pick_device():
    if FORCE_CPU or not torch.cuda.is_available(): return torch.device('cpu')
    try:
        x = torch.randn(16, 16, device='cuda'); (x @ x).sum().item()
        return torch.device('cuda')
    except Exception as e:
        print('CUDA smoke test failed, using CPU:', str(e)[:120]); return torch.device('cpu')
DEVICE = pick_device()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print('Device:', DEVICE, '| Variants:', list(VARIANTS))

Device: cuda | Variants: ['MLM+attn', 'noMLM+attn', 'MLM+mean', 'noMLM+mean']


## 2. Data, cleaning, split

In [27]:
df = pd.read_csv(CSV_PATH)
df = df.rename(columns={'Sentence': 'raw_text', 'intent': 'intent_label', 'Sentiment': 'sentiment_label'})
df.insert(0, 'uid', np.arange(len(df)))                      # stable id = original CSV row
df = df[['uid', 'raw_text', 'sentiment_label', 'intent_label']].dropna().copy()

GOLD_CSV  = '/content/annotation_with_sentiment_intent.csv'
gold      = pd.read_csv(GOLD_CSV, keep_default_na=False)
gold_uids = set(gold['uid'])
df['is_gold'] = df['uid'].isin(gold_uids)

def clean_text(text: str) -> str:
  text = str(text).lower().strip()
  text = re.sub(r'http\S+|www\S+', '', text)
  text = re.sub(
      r'[^\w\s\u0D80-\u0DFF\u200c\u200d?!.\U0001F300-\U0001FAFF\u2600-\u27BF]',
      ' ',
      text,
  )
  return re.sub(r'\s+', ' ', text).strip()

n0 = len(df)
df['text'] = df['raw_text'].apply(clean_text)
df = df[(df['text'].str.len() > 3) | df['is_gold']]           # never drop a gold row
n1 = len(df)

# dedupe: keep ALL gold rows; drop non-gold rows that duplicate each other or any gold text
g, rest = df[df['is_gold']], df[~df['is_gold']]
rest = rest[~rest['text'].isin(set(g['text']))].drop_duplicates('text')
df = pd.concat([g, rest]).sort_values('uid').reset_index(drop=True)

print(f'rows: {n0} -> {n1} (len filter) -> {len(df)} (dedupe)')
print('gold rows kept:', df['is_gold'].sum(), '/ 300')            # must be 300

sent_enc, intent_enc = LabelEncoder().fit(df['sentiment_label']), LabelEncoder().fit(df['intent_label'])
SENT_CLASSES, INTENT_CLASSES = list(sent_enc.classes_), list(intent_enc.classes_)
NS, NI = len(SENT_CLASSES), len(INTENT_CLASSES)
def add_ids(d):
    d = d.reset_index(drop=True).copy()
    d['sent_id']   = sent_enc.transform(d['sentiment_label'])
    d['intent_id'] = intent_enc.transform(d['intent_label'])
    return d
df = add_ids(df)
print('Sentiment:', SENT_CLASSES); print('Intent   :', INTENT_CLASSES)

rows: 5881 -> 5869 (len filter) -> 5834 (dedupe)
gold rows kept: 300 / 300
Sentiment: ['Conflict', 'Negative', 'Neutral', 'Positive']
Intent   : ['complaint', 'feedback', 'greeting', 'inquiry', 'other', 'suggestion']


In [28]:
def safe_split(d, size, key, seed):
    ok = d[key].value_counts().min() >= 2
    return train_test_split(d, test_size=size, random_state=seed, stratify=d[key] if ok else None)

test_df = df[df['uid'].isin(gold_uids)]
pool    = df[~df['uid'].isin(gold_uids)]
train_df, val_df = safe_split(pool, VAL_FRAC, 'intent_id', SPLIT_SEED)
train_df, val_df, test_df = [add_ids(d) for d in (train_df, val_df, test_df)]

assert len(test_df) == 300
assert not (set(train_df.uid) | set(val_df.uid)) & set(test_df.uid)
print(f'train {len(train_df)} | val {len(val_df)} | test {len(test_df)}')

for name, col in [('Sentiment', 'sentiment_label'), ('Intent', 'intent_label')]:
    t = pd.concat({k: d[col].value_counts() for k, d in [('train', train_df), ('val', val_df), ('test', test_df)]}, axis=1).fillna(0).astype(int)
    print(f'\n{name}\n{t}')
    low = t[t['test'] < 20]
    if len(low): print(f'  WARNING: <20 test rows (per-class F1 unreliable): {list(low.index)}')

train 4703 | val 831 | test 300

Sentiment
                 train  val  test
sentiment_label                  
Neutral           3103  560   184
Negative          1242  202    90
Positive           311   55    25
Conflict            47   14     1

Intent
              train  val  test
intent_label                  
inquiry        2577  455   167
feedback        623  110    39
greeting        463   82    27
complaint       433   77    28
suggestion      361   64    23
other           246   43    16


In [29]:
# ── Export blind annotation sheet: raw (uncleaned) text only, NO silver labels ──
if not os.path.exists(ANNOT_CSV):
    (test_df[['uid', 'raw_text']].sample(frac=1, random_state=0)
        .assign(gold_sentiment='', gold_intent='').to_csv(ANNOT_CSV, index=False))
    print('Wrote', ANNOT_CSV)
else:
    print('Annotation sheet already exists:', ANNOT_CSV)
print('Allowed gold_sentiment values:', SENT_CLASSES)
print('Allowed gold_intent values   :', INTENT_CLASSES)
print('Save the completed sheet as   :', GOLD_CSV, '(keep the uid column untouched)')

Annotation sheet already exists: /content//annotation_sheet.csv
Allowed gold_sentiment values: ['Conflict', 'Negative', 'Neutral', 'Positive']
Allowed gold_intent values   : ['complaint', 'feedback', 'greeting', 'inquiry', 'other', 'suggestion']
Save the completed sheet as   : /content/annotation_with_sentiment_intent.csv (keep the uid column untouched)


## 3. Shared components (datasets, models, metrics)

In [30]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, d):
        e = tok(d['text'].tolist(), truncation=True, max_length=MAX_LEN)
        self.ids, self.am = e['input_ids'], e['attention_mask']
        self.s, self.i = d['sent_id'].tolist(), d['intent_id'].tolist()
    def __len__(self): return len(self.ids)
    def __getitem__(self, k):
        return {'input_ids': self.ids[k], 'attention_mask': self.am[k], 'sent': self.s[k], 'intent': self.i[k]}

def collate(batch):
    out = tok.pad([{'input_ids': b['input_ids'], 'attention_mask': b['attention_mask']} for b in batch], return_tensors='pt')
    out['sent']   = torch.tensor([b['sent'] for b in batch])
    out['intent'] = torch.tensor([b['intent'] for b in batch])
    return out

train_ds, val_ds, test_ds = DS(train_df), DS(val_df), DS(test_df)
val_loader  = DataLoader(val_ds,  batch_size=FT_BATCH, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=FT_BATCH, shuffle=False, collate_fn=collate)

# ── MLM corpus: train + val text (+ optional extra). Test text is excluded. ──────
mlm_texts = train_df['text'].tolist() + val_df['text'].tolist()
if MLM_EXTRA_CSV:
    extra = pd.read_csv(MLM_EXTRA_CSV, usecols=[MLM_EXTRA_COL])[MLM_EXTRA_COL].dropna().map(clean_text)
    extra = extra[extra.str.len() > 3].drop_duplicates()
    extra = extra[~extra.isin(set(df['text']))]                 # drops test + labeled rows
    extra = extra.sample(min(MLM_EXTRA_N, len(extra)), random_state=SPLIT_SEED).tolist()
    mlm_texts += extra
    print(f'MLM extra rows: {len(extra)}')
assert not (set(mlm_texts) & set(test_df['text'])), 'test text leaked into MLM corpus'
_e = tok(mlm_texts, truncation=True, max_length=MAX_LEN, return_special_tokens_mask=True)
mlm_examples = [{'input_ids': a, 'attention_mask': b, 'special_tokens_mask': c}
                for a, b, c in zip(_e['input_ids'], _e['attention_mask'], _e['special_tokens_mask'])]
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=MLM_PROB)
print(f'MLM corpus: {len(mlm_examples)} texts')

MLM corpus: 5534 texts


In [31]:
class Pooler(nn.Module):
    # attn: one learned query per task over token states (multi-head); mean/cls: one shared vector.
    def __init__(self, hidden, mode, heads, dropout):
        super().__init__()
        self.mode = mode
        if mode == 'attn':
            self.attn = nn.MultiheadAttention(hidden, heads, dropout=dropout, batch_first=True)
            self.query_sent = nn.Parameter(torch.randn(1, 1, hidden))
            self.query_int  = nn.Parameter(torch.randn(1, 1, hidden))
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        if self.mode == 'attn':
            B, kpm = H.size(0), (mask == 0)
            outs = []
            for q in (self.query_sent, self.query_int):
                o, _ = self.attn(q.expand(B, -1, -1), H, H, key_padding_mask=kpm)
                outs.append(self.norm(o.squeeze(1)))
            return outs
        if self.mode == 'cls':
            p = H[:, 0]
        else:
            m = mask.unsqueeze(-1).to(H.dtype)
            p = (H * m).sum(1) / m.sum(1).clamp(min=1)
        p = self.norm(p)
        return p, p

class JointModel(nn.Module):
    def __init__(self, enc_path, pooling):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.pool = Pooler(h, pooling, NUM_ATTN_HEADS, DROPOUT)
        fc = lambda: nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.sent_fc, self.int_fc = fc(), fc()
        self.sent_head, self.intent_head = nn.Linear(h // 2, NS), nn.Linear(h // 2, NI)

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        ps, pi = self.pool(H, attention_mask)
        return self.sent_head(self.sent_fc(ps)), self.intent_head(self.int_fc(pi))

@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); S, I = [], []
    for b in loader:
        s, i = model(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))
        S.append(torch.softmax(s, -1).cpu()); I.append(torch.softmax(i, -1).cpu())
    return torch.cat(S).numpy(), torch.cat(I).numpy()

def class_weights(ids, K):
    c = np.bincount(ids, minlength=K).astype(float)
    return torch.tensor(np.where(c > 0, c.sum() / (K * np.maximum(c, 1)), 0.0), dtype=torch.float)

def macro_f1(y, p, K):
    # macro-F1 over classes present in y (gold)
    cm = np.bincount(np.asarray(y) * K + np.asarray(p), minlength=K * K).reshape(K, K)
    tp = np.diag(cm); den = cm.sum(0) + cm.sum(1)
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    return float(f1[cm.sum(1) > 0].mean())

def acc(y, p): return float((np.asarray(y) == np.asarray(p)).mean())

## 4. Training (MLM stage + fine-tuning) — resumable, caches to disk

In [32]:
def get_mlm_encoder(seed):
    out = f'{ENC_DIR}/mlm_seed{seed}'
    if os.path.exists(f'{out}/config.json'): return out
    set_seed(seed)
    m = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE); m.train()
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(mlm_examples, batch_size=MLM_BATCH, shuffle=True, collate_fn=mlm_collator, generator=g)
    opt = torch.optim.AdamW(m.parameters(), lr=MLM_LR, weight_decay=WEIGHT_DECAY)
    total = MLM_EPOCHS * math.ceil(len(loader) / MLM_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)
    for ep in range(MLM_EPOCHS):
        tl, n = 0.0, 0; opt.zero_grad()
        for step, b in enumerate(loader):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            if (b['labels'] != -100).any():                       # skip batches with nothing masked
                loss = m(**b).loss
                (loss / MLM_GRAD_ACC).backward(); tl += loss.item(); n += 1
            if (step + 1) % MLM_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
        print(f'  [MLM seed {seed}] epoch {ep+1}/{MLM_EPOCHS} loss {tl/max(n,1):.4f}')
    m.base_model.save_pretrained(out)
    del m, opt, sched; gc.collect(); torch.cuda.empty_cache()
    return out

W_SENT = class_weights(train_df['sent_id'].values, NS).to(DEVICE)
W_INT  = class_weights(train_df['intent_id'].values, NI).to(DEVICE)
val_ys, val_yi = val_df['sent_id'].values, val_df['intent_id'].values

def train_variant(name, use_mlm, pooling, seed):
    out = f'{PRED_DIR}/{name}__seed{seed}.npz'
    if os.path.exists(out): print(f'skip {name} seed {seed} (cached)'); return
    enc_path = get_mlm_encoder(seed) if use_mlm else MODEL_NAME
    set_seed(seed)
    model = JointModel(enc_path, pooling).to(DEVICE)
    g = torch.Generator(); g.manual_seed(seed)                    # same batch order across variants for a seed
    loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True, collate_fn=collate, generator=g)
    crit_s, crit_i = nn.CrossEntropyLoss(weight=W_SENT), nn.CrossEntropyLoss(weight=W_INT)
    head_params = [p for n, p in model.named_parameters() if not n.startswith('encoder.')]
    opt = torch.optim.AdamW([{'params': model.encoder.parameters(), 'lr': FT_LR * ENC_LR_MULT},
                             {'params': head_params, 'lr': FT_LR}], weight_decay=WEIGHT_DECAY)
    total = FT_EPOCHS * math.ceil(len(loader) / FT_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

    best, best_ep, best_state, bad = -1.0, 0, None, 0
    t0 = time.time()
    for ep in range(FT_EPOCHS):
        model.train(); tl = 0.0; opt.zero_grad()
        for step, b in enumerate(loader):
            s, i = model(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))
            loss = crit_s(s, b['sent'].to(DEVICE)) + crit_i(i, b['intent'].to(DEVICE))
            (loss / FT_GRAD_ACC).backward(); tl += loss.item()
            if (step + 1) % FT_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
        vs, vi = predict_probs(model, val_loader)
        f_s, f_i = macro_f1(val_ys, vs.argmax(1), NS), macro_f1(val_yi, vi.argmax(1), NI)
        score = (f_s + f_i) / 2
        print(f'  [{name} s{seed}] ep {ep+1:>2} | train {tl/len(loader):.4f} | val F1 sent {f_s:.3f} int {f_i:.3f} | mean {score:.3f}')
        if score > best:
            best, best_ep, bad = score, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: print('  early stop'); break
    model.load_state_dict(best_state)
    torch.save(best_state, f'{ENC_DIR}/{name}__seed{seed}.pt')
    vs, vi = predict_probs(model, val_loader); ts, ti = predict_probs(model, test_loader)
    np.savez(out, val_S=vs, val_I=vi, test_S=ts, test_I=ti, best_epoch=best_ep, best_val=best)
    print(f'done {name} seed {seed}: best epoch {best_ep}, val {best:.3f}, {time.time()-t0:.0f}s')
    del model, opt, sched, best_state; gc.collect(); torch.cuda.empty_cache()

In [33]:
for seed in SEEDS:
    for name, (use_mlm, pool) in VARIANTS.items():
        train_variant(name, use_mlm, pool, seed)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [MLM seed 42] epoch 1/3 loss 4.2701
  [MLM seed 42] epoch 2/3 loss 3.7690
  [MLM seed 42] epoch 3/3 loss 3.5649


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: /content//mlm_encoders/mlm_seed42
Key                 | Status  | 
--------------------+---------+-
pooler.dense.weight | MISSING | 
pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (38 outputs left out)

  [MLM+mean s44] ep  1 | train 3.0780 | val F1 sent 0.304 int 0.331 | mean 0.317
  [MLM+mean s44] ep  2 | train 2.4311 | val F1 sent 0.463 int 0.564 | mean 0.513
  [MLM+mean s44] ep  3 | train 2.0244 | val F1 sent 0.456 int 0.577 | mean 0.516
  [MLM+mean s44] ep  4 | train 1.8203 | val F1 sent 0.464 int 0.605 | mean 0.535
  [MLM+mean s44] ep  5 | train 1.7149 | val F1 sent 0.468 int 0.617 | mean 0.543
  [MLM+mean s44] ep  6 | train 1.6283 | val F1 sent 0.465 int 0.642 | mean 0.553
  [MLM+mean s44] ep  7 | train 1.5925 | val F1 sent 0.470 int 0.641 | mean 0.555
  [MLM+mean s44] ep  8 | train 1.5316 | val F1 sent 0.476 int 0.643 | mean 0.559
  [MLM+mean s44] ep  9 | train 1.4928 | val F1 sent 0.466 int 0.656 | mean 0.561
  [MLM+mean s44] ep 10 | train 1.4807 | val F1 sent 0.470 int 0.653 | mean 0.561
done MLM+mean seed 44: best epoch 10, val 0.561, 263s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [noMLM+mean s44] ep  1 | train 3.1025 | val F1 sent 0.245 int 0.264 | mean 0.254
  [noMLM+mean s44] ep  2 | train 2.6613 | val F1 sent 0.391 int 0.466 | mean 0.428
  [noMLM+mean s44] ep  3 | train 2.1999 | val F1 sent 0.451 int 0.543 | mean 0.497
  [noMLM+mean s44] ep  4 | train 1.9712 | val F1 sent 0.450 int 0.572 | mean 0.511
  [noMLM+mean s44] ep  5 | train 1.8469 | val F1 sent 0.469 int 0.594 | mean 0.532
  [noMLM+mean s44] ep  6 | train 1.7517 | val F1 sent 0.470 int 0.607 | mean 0.538
  [noMLM+mean s44] ep  7 | train 1.7167 | val F1 sent 0.475 int 0.619 | mean 0.547
  [noMLM+mean s44] ep  8 | train 1.6521 | val F1 sent 0.471 int 0.619 | mean 0.545
  [noMLM+mean s44] ep  9 | train 1.6039 | val F1 sent 0.475 int 0.634 | mean 0.554
  [noMLM+mean s44] ep 10 | train 1.6023 | val F1 sent 0.475 int 0.632 | mean 0.554
done noMLM+mean seed 44: best epoch 9, val 0.554, 261s


## 5. Evaluation

In [34]:
RUNS = {}
for name in VARIANTS:
    for s in SEEDS:
        f = f'{PRED_DIR}/{name}__seed{s}.npz'
        if os.path.exists(f): RUNS[(name, s)] = dict(np.load(f))
print(f'loaded {len(RUNS)}/{len(VARIANTS)*len(SEEDS)} runs')

def P(name, s, split, mask=None):
    r = RUNS[(name, s)]; a, b = r[f'{split}_S'].argmax(1), r[f'{split}_I'].argmax(1)
    return (a, b) if mask is None else (a[mask], b[mask])

def results_table(split, ys, yi, mask=None):
    rows = []
    for name in VARIANTS:
        per = []
        for s in SEEDS:
            if (name, s) not in RUNS: continue
            ps, pi = P(name, s, split, mask)
            per.append([acc(ys, ps), macro_f1(ys, ps, NS), acc(yi, pi), macro_f1(yi, pi, NI), float(((ys == ps) & (yi == pi)).mean())])
        a = np.array(per)
        rows.append({'variant': name, 'n_seeds': len(a),
                     **{c: f'{a[:, j].mean():.3f} ± {a[:, j].std():.3f}'
                        for j, c in enumerate(['sent acc', 'sent F1', 'intent acc', 'intent F1', 'joint acc'])}})
    return pd.DataFrame(rows).set_index('variant')

print('VAL (silver labels) — mean ± std over seeds'); display(results_table('val', val_ys, val_yi))
print('\nTEST vs SILVER labels (proxy only, not the headline)')
display(results_table('test', test_df['sent_id'].values, test_df['intent_id'].values))

loaded 12/12 runs
VAL (silver labels) — mean ± std over seeds


,n_seeds,sent acc,sent F1,intent acc,intent F1,joint acc
variant,,,,,,
MLM+attn,3,0.719 ± 0.006,0.508 ± 0.011,0.779 ± 0.005,0.673 ± 0.009,0.573 ± 0.005
noMLM+attn,3,0.705 ± 0.004,0.541 ± 0.009,0.765 ± 0.016,0.657 ± 0.024,0.556 ± 0.007
MLM+mean,3,0.709 ± 0.010,0.480 ± 0.008,0.765 ± 0.002,0.656 ± 0.003,0.558 ± 0.011
noMLM+mean,3,0.699 ± 0.007,0.477 ± 0.004,0.750 ± 0.002,0.638 ± 0.008,0.543 ± 0.004



TEST vs SILVER labels (proxy only, not the headline)


,n_seeds,sent acc,sent F1,intent acc,intent F1,joint acc
variant,,,,,,
MLM+attn,3,0.751 ± 0.002,0.512 ± 0.005,0.806 ± 0.011,0.699 ± 0.023,0.616 ± 0.011
noMLM+attn,3,0.722 ± 0.007,0.488 ± 0.003,0.773 ± 0.014,0.647 ± 0.019,0.563 ± 0.011
MLM+mean,3,0.738 ± 0.008,0.500 ± 0.007,0.783 ± 0.015,0.672 ± 0.029,0.592 ± 0.015
noMLM+mean,3,0.728 ± 0.007,0.494 ± 0.006,0.790 ± 0.014,0.668 ± 0.019,0.584 ± 0.010


In [35]:
GOLD_CSV = '/content/annotation_with_sentiment_intent.csv'

In [36]:
import os

print("--- Checking Root Input ---")
!find /content/ -name "annotation_with_sentiment_intent.csv"

--- Checking Root Input ---
/content/annotation_with_sentiment_intent.csv


In [37]:
import pandas as pd

path = '/content/annotation_with_sentiment_intent.csv'

df = pd.read_csv(path, keep_default_na=False, dtype=str)
print("File read successfully!")
print("Columns found:", df.columns.tolist())
print("Total rows:", len(df))

# Check for required columns
expected_cols = {'uid', 'gold_sentiment', 'gold_intent'}
missing_cols = expected_cols - set(df.columns)
if missing_cols:
    print(f"❌ Missing expected columns: {missing_cols}")
else:
    # Check populated rows
    valid = df[(df['gold_sentiment'].str.strip() != '') & (df['gold_intent'].str.strip() != '')]
    print(f"Rows with non-empty annotations: {len(valid)} / {len(df)}")

    # Check classes against notebook classes
    print("\nUnique sentiment labels in CSV:", df['gold_sentiment'].unique())
    print("Unique intent labels in CSV:", df['gold_intent'].unique())

File read successfully!
Columns found: ['uid', 'raw_text', 'gold_sentiment', 'gold_intent']
Total rows: 300
Rows with non-empty annotations: 300 / 300

Unique sentiment labels in CSV: ['Neutral' 'Negative' 'Positive' 'Conflict']
Unique intent labels in CSV: ['suggestion' 'inquiry' 'feedback' 'other' 'greeting' 'complaint']


In [38]:
# ── Gold test set ─────────────────────────────────────────────────────────────
def load_gold(path):
    if not os.path.exists(path): return None
    g = pd.read_csv(path, keep_default_na=False, dtype=str)
    for c in ('gold_sentiment', 'gold_intent'): g[c] = g[c].str.strip()
    g = g[(g['gold_sentiment'] != '') & (g['gold_intent'] != '')]
    bad_s, bad_i = set(g['gold_sentiment']) - set(SENT_CLASSES), set(g['gold_intent']) - set(INTENT_CLASSES)
    if bad_s or bad_i: raise ValueError(f'Unknown labels -> sentiment: {bad_s} intent: {bad_i}')
    g['uid'] = g['uid'].astype(int)
    m = test_df[['uid']].merge(g[['uid', 'gold_sentiment', 'gold_intent']], on='uid', how='left')
    mask = m['gold_sentiment'].notna().values
    return (mask, sent_enc.transform(m.loc[mask, 'gold_sentiment']), intent_enc.transform(m.loc[mask, 'gold_intent']))

GOLD = load_gold(GOLD_CSV)
if GOLD is None:
    print(f'No gold file at {GOLD_CSV} yet — annotate {ANNOT_CSV}, save as that name, re-run this cell.')
else:
    G_MASK, GS, GI = GOLD
    print(f'Gold rows annotated: {G_MASK.sum()}/{len(test_df)}')
    if G_MASK.sum() < 300: print('WARNING: fewer than 300 gold rows')
    # human vs silver agreement (how noisy is the silver supervision?)
    ss, si = test_df.loc[G_MASK, 'sent_id'].values, test_df.loc[G_MASK, 'intent_id'].values
    print(f'gold vs silver | sentiment acc {acc(GS, ss):.3f} κ {cohen_kappa_score(GS, ss):.3f} | '
          f'intent acc {acc(GI, si):.3f} κ {cohen_kappa_score(GI, si):.3f}')
    print('\nTEST (GOLD) — mean ± std over seeds')
    display(results_table('test', GS, GI, G_MASK))

Gold rows annotated: 300/300
gold vs silver | sentiment acc 1.000 κ 1.000 | intent acc 0.450 κ 0.293

TEST (GOLD) — mean ± std over seeds


,n_seeds,sent acc,sent F1,intent acc,intent F1,joint acc
variant,,,,,,
MLM+attn,3,0.751 ± 0.002,0.512 ± 0.005,0.474 ± 0.004,0.410 ± 0.010,0.401 ± 0.007
noMLM+attn,3,0.722 ± 0.007,0.488 ± 0.003,0.476 ± 0.017,0.412 ± 0.027,0.394 ± 0.016
MLM+mean,3,0.738 ± 0.008,0.500 ± 0.007,0.484 ± 0.002,0.429 ± 0.006,0.410 ± 0.003
noMLM+mean,3,0.728 ± 0.007,0.494 ± 0.006,0.469 ± 0.006,0.407 ± 0.007,0.390 ± 0.003


In [39]:
# ── Paired bootstrap on gold test rows: resample rows, average macro-F1 over seeds ──
def bootstrap(ys, yi, mask, B=1000, seed=0):
    n = len(ys); idx = np.random.default_rng(seed).integers(0, n, (B, n))
    boot, point = {}, {}
    for name in VARIANTS:
        preds = [P(name, s, 'test', mask) for s in SEEDS if (name, s) in RUNS]
        point[name] = np.array([np.mean([macro_f1(ys, a, NS) for a, _ in preds]),
                                np.mean([macro_f1(yi, b, NI) for _, b in preds])])
        fs = np.zeros((B, 2))
        for k in range(B):
            ix = idx[k]
            fs[k, 0] = np.mean([macro_f1(ys[ix], a[ix], NS) for a, _ in preds])
            fs[k, 1] = np.mean([macro_f1(yi[ix], b[ix], NI) for _, b in preds])
        boot[name] = fs
    return boot, point

def contrasts():
    mlm = lambda p: f'MLM+{p}'; no = lambda p: f'noMLM+{p}'
    out = [('MLM effect (avg over pooling)', [mlm(p) for p in POOLINGS], [no(p) for p in POOLINGS])]
    out += [(f'MLM effect | {p}', [mlm(p)], [no(p)]) for p in POOLINGS]
    for a, b in itertools.combinations(POOLINGS, 2):
        out.append((f'{a} − {b} (avg over MLM)', [mlm(a), no(a)], [mlm(b), no(b)]))
        out += [(f'{a} − {b} | MLM', [mlm(a)], [mlm(b)]), (f'{a} − {b} | noMLM', [no(a)], [no(b)])]
    return out

if GOLD is not None:
    boot, point = bootstrap(GS, GI, G_MASK)
    print('Macro-F1 95% CI per variant (seed-averaged, bootstrap over test rows)')
    for name in VARIANTS:
        lo, hi = np.percentile(boot[name], [2.5, 97.5], axis=0)
        print(f'  {name:<14} sent {point[name][0]:.3f} [{lo[0]:.3f}, {hi[0]:.3f}] | intent {point[name][1]:.3f} [{lo[1]:.3f}, {hi[1]:.3f}]')
    rows = []
    for label, plus, minus in contrasts():
        d_b = np.mean([boot[v] for v in plus], 0) - np.mean([boot[v] for v in minus], 0)
        d_p = np.mean([point[v] for v in plus], 0) - np.mean([point[v] for v in minus], 0)
        lo, hi = np.percentile(d_b, [2.5, 97.5], axis=0)
        for t, task in enumerate(('sentiment', 'intent')):
            rows.append({'contrast': label, 'task': task, 'Δ macro-F1': round(d_p[t], 4),
                         '95% CI': f'[{lo[t]:+.3f}, {hi[t]:+.3f}]', 'CI excludes 0': bool(lo[t] > 0 or hi[t] < 0)})
    display(pd.DataFrame(rows))
    print('CI covers test-set sampling noise only (seed spread is in the std columns above).')

Macro-F1 95% CI per variant (seed-averaged, bootstrap over test rows)
  MLM+attn       sent 0.512 [0.462, 0.728] | intent 0.410 [0.352, 0.462]
  noMLM+attn     sent 0.488 [0.437, 0.698] | intent 0.412 [0.354, 0.465]
  MLM+mean       sent 0.500 [0.452, 0.713] | intent 0.429 [0.367, 0.481]
  noMLM+mean     sent 0.494 [0.443, 0.708] | intent 0.407 [0.351, 0.458]


,contrast,task,Δ macro-F1,95% CI,CI excludes 0
0,MLM effect (avg over pooling),sentiment,0.0145,"[-0.007, +0.042]",False
1,MLM effect (avg over pooling),intent,0.0100,"[-0.018, +0.038]",False
2,MLM effect | attn,sentiment,0.0237,"[+0.004, +0.057]",True
3,MLM effect | attn,intent,-0.0016,"[-0.036, +0.035]",False
4,MLM effect | mean,sentiment,0.0052,"[-0.024, +0.037]",False
5,MLM effect | mean,intent,0.0215,"[-0.008, +0.051]",False
6,attn − mean (avg over MLM),sentiment,0.0029,"[-0.012, +0.018]",False
7,attn − mean (avg over MLM),intent,-0.0071,"[-0.026, +0.013]",False
8,attn − mean | MLM,sentiment,0.0121,"[-0.006, +0.035]",False
9,attn − mean | MLM,intent,-0.0186,"[-0.044, +0.007]",False


CI covers test-set sampling noise only (seed spread is in the std columns above).


In [40]:
# ── Per-class report (first seed) ─────────────────────────────────────────────
if GOLD is not None:
    for name in VARIANTS:
        ps, pi = P(name, SEEDS[0], 'test', G_MASK)
        print(f'\n===== {name} (seed {SEEDS[0]}) =====')
        print('-- sentiment'); print(classification_report(GS, ps, labels=range(NS), target_names=SENT_CLASSES, digits=3, zero_division=0))
        print('-- intent');    print(classification_report(GI, pi, labels=range(NI), target_names=INTENT_CLASSES, digits=3, zero_division=0))


===== MLM+attn (seed 42) =====
-- sentiment
              precision    recall  f1-score   support

    Conflict      0.000     0.000     0.000         1
    Negative      0.685     0.844     0.756        90
     Neutral      0.851     0.745     0.794       184
    Positive      0.500     0.520     0.510        25

    accuracy                          0.753       300
   macro avg      0.509     0.527     0.515       300
weighted avg      0.769     0.753     0.756       300

-- intent
              precision    recall  f1-score   support

   complaint      0.545     0.295     0.383        61
    feedback      0.488     0.213     0.296        94
    greeting      0.517     0.652     0.577        23
     inquiry      0.490     0.962     0.649        78
       other      0.143     0.154     0.148        13
  suggestion      0.467     0.452     0.459        31

    accuracy                          0.480       300
   macro avg      0.442     0.455     0.419       300
weighted avg      0.48

In [41]:
def load_model(name, seed):
    use_mlm, pool = VARIANTS[name]
    m = JointModel(MODEL_NAME, pool).to(DEVICE)
    m.load_state_dict(torch.load(f'{ENC_DIR}/{name}__seed{seed}.pt', map_location=DEVICE))
    return m.eval()

@torch.no_grad()
def predict_text(model, texts):
    cl = [clean_text(t) for t in texts]
    e = tok(cl, truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)
    s, i = model(e['input_ids'], e['attention_mask'])
    ps, pi = torch.softmax(s, -1).cpu().numpy(), torch.softmax(i, -1).cpu().numpy()
    return pd.DataFrame({
        'text': texts,
        'sentiment': [SENT_CLASSES[k] for k in ps.argmax(1)], 'sent_conf': ps.max(1).round(3),
        'intent': [INTENT_CLASSES[k] for k in pi.argmax(1)], 'intent_conf': pi.max(1).round(3)})

In [43]:
model = load_model('MLM+attn', 42)
display(predict_text(model, [
    "Subha aluth awuruddak wewa hemotama! Any offers today?",
    "Data package eka activate une na, salli kapala thiyeddi ai mehema wenne??",
    "Apo mehema service ekak nam minissunta epa wenawa.."
]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,text,sentiment,sent_conf,intent,intent_conf
0,Subha aluth awuruddak wewa hemotama! Any offer...,Neutral,0.717,greeting,0.690
1,"Data package eka activate une na, salli kapala...",Neutral,0.580,inquiry,0.973
2,Apo mehema service ekak nam minissunta epa wen...,Negative,0.710,inquiry,0.937
